In [1]:
import numpy as np
import matplotlib.pyplot as plt
from jinja2.compiler import has_safe_repr
from matplotlib.colors import ListedColormap
import random

class Maze:
    def __init__(self):
        self.grid = np.array([
            [0, 0, 0, 0, 0],
            [0, 0, 1, 0, 0],
            [2, 0, 1, 0, 3]
        ])

        self.start = (2, 0)
        self.goal = (2, 4)
        self.position = self.start

        self.actions = {
            "up": (-1, 0),
            "down": (1, 0),
            "left": (0, -1),
            "right": (0, 1)
        }

        # where 2 is start point and 3 is end point
        # and 1 is the wall
        # and 0 is the free space

    def reset(self):
        self.start = (2,0)
        return self.position

    def is_valid(self, position):
        row, col = position

        return (
            0 <= row < self.grid.shape[0]
            and 0 <= col < self.grid.shape[1]
            and self.grid[row, col] != 1
        )

    def step(self, action):
        if action not in self.actions:
            raise ValueError(f"Invalid action: {action}")

        dr, dc = self.actions[action]
        row, col = self.position

        new_state = (row + dr, col + dc)

        if not self.is_valid(new_state):
            return self.position, -2, False

        self.position = new_state

        if self.position == self.goal:
            return self.position, 0, True

        return self.position, -1, False



    def display(self):
        cmap = ListedColormap([
            "white",  # 0 = Empty
            "black",  # 1 = Wall
            "gold",   # 2 = Start
            "lime"    # 3 = Goal
        ])

        fig, ax = plt.subplots(figsize=(8, 5))

        ax.imshow(self.grid, cmap=cmap, vmin=0, vmax=3)

        # Draw agent at current position
        row, col = self.position
        ax.scatter(col, row, color="red", s=250, zorder=5)

        # Grid coordinates
        rows, cols = self.grid.shape

        ax.set_xticks(np.arange(cols))
        ax.set_yticks(np.arange(rows))

        # Grid borders
        ax.set_xticks(np.arange(-0.5, cols, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, rows, 1), minor=True)

        ax.grid(which="minor", color="black", linewidth=2)
        ax.tick_params(which="minor", bottom=False, left=False)

        ax.xaxis.tick_top()
        ax.set_title(f"Agent Position: {self.position}")

        plt.show()

In [9]:
class Agent:
    def __init__(self, maze):
        self.maze = maze
        self.episode_history = {} # stores each episode
        self.actions = ["up", "down", "left", "right"]
        self.gamma = 0.9

    def choose_action(self):
        return random.choice(self.actions)

    def generate_episode(self, max_steps=1000):
        # Reset maze to starting position
        self.maze.position = self.maze.start
        self.history = []

        state = self.maze.position

        for step in range(max_steps):
            # Choose an action
            action = self.choose_action()

            # Execute the action
            next_state, reward, done = self.maze.step(action)

            # Record transition
            self.history.append({
                "state": state,
                "action": action,
                "next_state": next_state,
                "reward": reward,
                "done": done
            })

            # Update current state
            state = next_state

            # Stop if goal reached
            if done:
                break

        # Automatically assign episode index
        episode_id = len(self.episode_history)

        # Store a separate copy of the episode
        self.episode_history[episode_id] = [
            move.copy() for move in self.history
        ]

        return self.history

    def calculate_return(self, episode_id):
        episode = self.episode_history[episode_id]
        G = 0.0

        for t, move in enumerate(reversed(episode)):
            G += (self.gamma ** t) * move["reward"]

        return G


In [10]:
maze = Maze()

In [11]:
agent = Agent(maze)

In [20]:
agent.generate_episode(max_steps=1000)

[{'state': (2, 0),
  'action': 'right',
  'next_state': (2, 1),
  'reward': -1,
  'done': False},
 {'state': (2, 1),
  'action': 'left',
  'next_state': (2, 0),
  'reward': -1,
  'done': False},
 {'state': (2, 0),
  'action': 'down',
  'next_state': (2, 0),
  'reward': -2,
  'done': False},
 {'state': (2, 0),
  'action': 'up',
  'next_state': (1, 0),
  'reward': -1,
  'done': False},
 {'state': (1, 0),
  'action': 'up',
  'next_state': (0, 0),
  'reward': -1,
  'done': False},
 {'state': (0, 0),
  'action': 'down',
  'next_state': (1, 0),
  'reward': -1,
  'done': False},
 {'state': (1, 0),
  'action': 'down',
  'next_state': (2, 0),
  'reward': -1,
  'done': False},
 {'state': (2, 0),
  'action': 'right',
  'next_state': (2, 1),
  'reward': -1,
  'done': False},
 {'state': (2, 1),
  'action': 'down',
  'next_state': (2, 1),
  'reward': -2,
  'done': False},
 {'state': (2, 1),
  'action': 'down',
  'next_state': (2, 1),
  'reward': -2,
  'done': False},
 {'state': (2, 1),
  'action': '

In [21]:
agent.calculate_return(1)

-11.434559833218023